# Adaptive Router Experiment: GSM8K Evaluation

This notebook evaluates the **Adaptive Inference Router** on 200 GSM8K math problems.

## System Components

- **Task 2**: Adaptive Router (H_int-based routing)
- **Task 3**: Option-Normalized Entropy (for MC questions)
- **Task 4**: Constrained Decoding (for MC questions)

## Experiment Design

Compare 4 strategies:
1. **Always-Direct**: Use direct answer for all problems
2. **Always-CoT**: Use Chain-of-Thought for all problems
3. **Random**: Randomly choose between direct and CoT
4. **Adaptive Router**: Use entropy-based routing (our approach)

## Metrics

- **Accuracy**: % correct answers
- **Total Tokens**: Sum of all tokens used
- **Avg Tokens/Query**: Average tokens per problem
- **Efficiency**: Accuracy per 1000 tokens
- **Token Savings**: % reduction vs Always-CoT

## 1. Setup and Imports

In [ ]:
import sys
import subprocess

print("=" * 70)
print("SETUP - Adaptive Router Full Experiment")
print("=" * 70)

# 1. Clean and clone
print("\n[1/3] Cloning repository...")
subprocess.run(["rm", "-rf", "/content/intention-collapse-experiments"],
               capture_output=True, check=False)
result = subprocess.run(
    ["git", "clone", "-q",
     "https://github.com/patriciomvera/intention-collapse-experiments.git",
     "/content/intention-collapse-experiments"],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(f"Git clone failed: {result.stderr}")
print("   ✓ Repository cloned")

# 2. Install package
print("[2/3] Installing intention-collapse package...")
result = subprocess.run(
    ["pip", "install", "-q", "-e", "/content/intention-collapse-experiments/"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"   ⚠ pip warnings:\n{result.stderr[-500:]}")
else:
    print("   ✓ Package installed")

# 3. Add to path
print("[3/3] Configuring path...")
REPO = "/content/intention-collapse-experiments"
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print("   ✓ Path configured")

print("\n" + "=" * 70)

import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import json
import random
from collections import defaultdict
from datetime import datetime

from src.router import AdaptiveInferenceRouter, RouteDecision
from src.metrics import compute_option_normalized_entropy, compute_entropy_decomposition

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("✓ Imports successful")
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
import sys
import subprocess

print("=" * 70)
print("SETUP - Adaptive Router Full Experiment")
print("=" * 70)

# 1. Clean and clone
print("\n[1/4] Cloning repository...")
subprocess.run(["rm", "-rf", "/content/intention-collapse-experiments"],
               capture_output=True, check=False)
result = subprocess.run(
    ["git", "clone", "-q",
     "https://github.com/patriciomvera/intention-collapse-experiments.git",
     "/content/intention-collapse-experiments"],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(f"Git clone failed: {result.stderr}")
print("   ✓ Repository cloned")

# 2. Install bitsandbytes (required for 4-bit quantization)
print("[2/4] Installing bitsandbytes...")
subprocess.run(
    ["pip", "install", "-q", "-U", "bitsandbytes>=0.46.1"],
    capture_output=True, text=True
)
print("   ✓ bitsandbytes installed")

# 3. Install package
print("[3/4] Installing intention-collapse package...")
result = subprocess.run(
    ["pip", "install", "-q", "-e", "/content/intention-collapse-experiments/"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"   ⚠ pip warnings:\n{result.stderr[-500:]}")
else:
    print("   ✓ Package installed")

# 4. Add to path
print("[4/4] Configuring path...")
REPO = "/content/intention-collapse-experiments"
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print("   ✓ Path configured")

print("\n" + "=" * 70)

import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import json
import random
from collections import defaultdict
from datetime import datetime

from src.router import AdaptiveInferenceRouter, RouteDecision
from src.metrics import compute_option_normalized_entropy, compute_entropy_decomposition

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("✓ Imports successful")
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
# Experiment configuration
CONFIG = {
    # Model
    'model_name': 'Qwen/Qwen2.5-7B-Instruct',  # or 'mistralai/Mistral-7B-Instruct-v0.3'
    'quantization': '4bit',
    
    # Dataset
    'benchmark': 'gsm8k',
    'n_problems': 200,
    'seed': 42,
    
    # Router thresholds
    'entropy_threshold_low': 0.5,
    'entropy_threshold_high': 1.2,
    
    # Generation parameters
    'max_tokens_direct': 50,
    'max_tokens_cot': 512,
    
    # Output
    'results_dir': '../results/router_experiment',
    'save_intermediate': True,
    'verbose': False
}

# Create results directory
os.makedirs(CONFIG['results_dir'], exist_ok=True)

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 3. Load Model and Tokenizer

In [ ]:
print(f"Loading model: {CONFIG['model_name']}...")

# Quantization config
if CONFIG['quantization'] == '4bit':
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    print("  Using 4-bit quantization")
elif CONFIG['quantization'] == '8bit':
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)
    print("  Using 8-bit quantization")
else:
    quantization_config = None
    print("  Using full precision")

# Load model
model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_name'],
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG['model_name'],
    trust_remote_code=True
)

print(f"✓ Model loaded on {next(model.parameters()).device}")
print(f"✓ Model parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## 4. Load GSM8K Dataset

In [ ]:
print(f"Loading GSM8K dataset ({CONFIG['n_problems']} problems)...")

# Load full dataset
dataset = load_dataset("gsm8k", "main", split="test")

# Sample consistent subset
random.seed(CONFIG['seed'])
indices = sorted(random.sample(range(len(dataset)), CONFIG['n_problems']))

# Extract problems
problems = []
for idx in indices:
    item = dataset[idx]
    # Extract numerical answer from "#### X" format
    answer_parts = item['answer'].split('####')
    if len(answer_parts) == 2:
        ground_truth = answer_parts[1].strip()
    else:
        # Fallback: take last number
        import re
        numbers = re.findall(r'-?\d+\.?\d*', item['answer'])
        ground_truth = numbers[-1] if numbers else ""
    
    problems.append({
        'idx': idx,
        'question': item['question'],
        'full_answer': item['answer'],
        'ground_truth': ground_truth
    })

print(f"✓ Loaded {len(problems)} problems")
print(f"\nExample problem:")
print(f"  Question: {problems[0]['question'][:100]}...")
print(f"  Ground truth: {problems[0]['ground_truth']}")

## 5. Define Evaluation Function

In [ ]:
def evaluate_strategy(strategy_name, problems, router=None, force_route=None, verbose=False):
    """
    Evaluate a routing strategy on problems.
    
    Args:
        strategy_name: Name of strategy
        problems: List of problem dicts
        router: AdaptiveInferenceRouter instance
        force_route: RouteDecision to force (for baselines)
        verbose: Print progress
    
    Returns:
        Dictionary with results
    """
    results = []
    total_tokens = 0
    correct_count = 0
    
    # Track routing decisions (for adaptive)
    route_counts = defaultdict(int)
    entropy_history = []
    
    print(f"\nEvaluating: {strategy_name}")
    print("=" * 60)
    
    for i, problem in enumerate(tqdm(problems, desc=strategy_name)):
        try:
            # Generate answer
            result = router.generate(
                question=problem['question'],
                ground_truth=problem['ground_truth'],
                force_route=force_route
            )
            
            # Track metrics
            total_tokens += result.total_tokens
            if result.is_correct:
                correct_count += 1
            
            route_counts[result.route_taken.value] += 1
            entropy_history.append(result.intention_entropy)
            
            # Store result
            results.append({
                'problem_idx': i,
                'question': problem['question'][:100] + '...',
                'ground_truth': problem['ground_truth'],
                'extracted_answer': result.extracted_answer,
                'is_correct': result.is_correct,
                'route_taken': result.route_taken.value,
                'entropy': result.intention_entropy,
                'total_tokens': result.total_tokens,
                'input_tokens': result.input_tokens,
                'output_tokens': result.output_tokens
            })
            
            if verbose and i < 3:
                print(f"\n  Problem {i+1}:")
                print(f"    Route: {result.route_taken.value}")
                print(f"    Entropy: {result.intention_entropy:.3f}")
                print(f"    Answer: {result.extracted_answer}")
                print(f"    Correct: {result.is_correct}")
                print(f"    Tokens: {result.total_tokens}")
        
        except Exception as e:
            print(f"\n  Error on problem {i}: {e}")
            results.append({
                'problem_idx': i,
                'question': problem['question'][:100] + '...',
                'ground_truth': problem['ground_truth'],
                'extracted_answer': '',
                'is_correct': False,
                'route_taken': 'error',
                'entropy': float('nan'),
                'total_tokens': 0,
                'input_tokens': 0,
                'output_tokens': 0
            })
    
    # Calculate metrics
    accuracy = correct_count / len(problems)
    avg_tokens = total_tokens / len(problems)
    efficiency = accuracy / (total_tokens / 1000)  # Accuracy per 1k tokens
    
    # Summary
    summary = {
        'strategy': strategy_name,
        'n_problems': len(problems),
        'correct': correct_count,
        'accuracy': accuracy,
        'total_tokens': total_tokens,
        'avg_tokens': avg_tokens,
        'efficiency': efficiency,
        'route_distribution': dict(route_counts),
        'entropy_mean': np.nanmean(entropy_history),
        'entropy_std': np.nanstd(entropy_history)
    }
    
    print(f"\n  Results:")
    print(f"    Accuracy: {accuracy:.1%} ({correct_count}/{len(problems)})")
    print(f"    Total tokens: {total_tokens:,}")
    print(f"    Avg tokens/query: {avg_tokens:.1f}")
    print(f"    Efficiency: {efficiency:.4f}")
    print(f"    Route distribution: {dict(route_counts)}")
    
    return {
        'summary': summary,
        'results': results
    }

## 6. Run Experiments

### 6.1 Initialize Router

In [ ]:
print("Initializing Adaptive Router...")

router = AdaptiveInferenceRouter(
    model=model,
    tokenizer=tokenizer,
    entropy_threshold_low=CONFIG['entropy_threshold_low'],
    entropy_threshold_high=CONFIG['entropy_threshold_high'],
    benchmark=CONFIG['benchmark'],
    verbose=CONFIG['verbose']
)

print("✓ Router initialized")
print(f"  Thresholds: low={CONFIG['entropy_threshold_low']}, high={CONFIG['entropy_threshold_high']}")

### 6.2 Strategy 1: Always Direct

In [ ]:
router.reset_statistics()
results_always_direct = evaluate_strategy(
    "Always-Direct",
    problems,
    router=router,
    force_route=RouteDecision.DIRECT,
    verbose=True
)

### 6.3 Strategy 2: Always CoT

In [ ]:
router.reset_statistics()
results_always_cot = evaluate_strategy(
    "Always-CoT",
    problems,
    router=router,
    force_route=RouteDecision.COT,
    verbose=True
)

### 6.4 Strategy 3: Random Routing

In [ ]:
# Random routing: for each problem, randomly choose direct or CoT
router.reset_statistics()

results_random = []
total_tokens_random = 0
correct_count_random = 0
route_counts_random = defaultdict(int)

print("\nEvaluating: Random Routing")
print("=" * 60)

random.seed(CONFIG['seed'])
for i, problem in enumerate(tqdm(problems, desc="Random Routing")):
    try:
        # Randomly choose route
        force_route = random.choice([RouteDecision.DIRECT, RouteDecision.COT])
        
        result = router.generate(
            question=problem['question'],
            ground_truth=problem['ground_truth'],
            force_route=force_route
        )
        
        total_tokens_random += result.total_tokens
        if result.is_correct:
            correct_count_random += 1
        
        route_counts_random[result.route_taken.value] += 1
        
        results_random.append({
            'problem_idx': i,
            'question': problem['question'][:100] + '...',
            'ground_truth': problem['ground_truth'],
            'extracted_answer': result.extracted_answer,
            'is_correct': result.is_correct,
            'route_taken': result.route_taken.value,
            'entropy': result.intention_entropy,
            'total_tokens': result.total_tokens
        })
    except Exception as e:
        print(f"\n  Error on problem {i}: {e}")
        results_random.append({
            'problem_idx': i,
            'is_correct': False,
            'total_tokens': 0
        })

accuracy_random = correct_count_random / len(problems)
avg_tokens_random = total_tokens_random / len(problems)
efficiency_random = accuracy_random / (total_tokens_random / 1000)

results_random_dict = {
    'summary': {
        'strategy': 'Random',
        'accuracy': accuracy_random,
        'total_tokens': total_tokens_random,
        'avg_tokens': avg_tokens_random,
        'efficiency': efficiency_random,
        'route_distribution': dict(route_counts_random)
    },
    'results': results_random
}

print(f"\n  Results:")
print(f"    Accuracy: {accuracy_random:.1%}")
print(f"    Total tokens: {total_tokens_random:,}")
print(f"    Avg tokens/query: {avg_tokens_random:.1f}")
print(f"    Efficiency: {efficiency_random:.4f}")

### 6.5 Strategy 4: Adaptive Router (Our Approach)

In [ ]:
router.reset_statistics()
results_adaptive = evaluate_strategy(
    "Adaptive-Router",
    problems,
    router=router,
    force_route=None,  # Let router decide!
    verbose=True
)

## 7. Results Comparison

In [ ]:
# Compile results
all_results = [
    results_always_direct['summary'],
    results_always_cot['summary'],
    results_random_dict['summary'],
    results_adaptive['summary']
]

# Create comparison DataFrame
df_comparison = pd.DataFrame(all_results)
df_comparison = df_comparison[['strategy', 'accuracy', 'total_tokens', 'avg_tokens', 'efficiency']]

print("\n" + "="*80)
print("RESULTS COMPARISON")
print("="*80)
print(df_comparison.to_string(index=False))
print("="*80)

# Calculate improvements
baseline_cot = results_always_cot['summary']
adaptive = results_adaptive['summary']

accuracy_diff = (adaptive['accuracy'] - baseline_cot['accuracy']) * 100
token_savings = (1 - adaptive['total_tokens'] / baseline_cot['total_tokens']) * 100
efficiency_improvement = (adaptive['efficiency'] / baseline_cot['efficiency'] - 1) * 100

print("\n📊 ADAPTIVE ROUTER vs ALWAYS-COT:")
print(f"  Accuracy difference: {accuracy_diff:+.1f} percentage points")
print(f"  Token savings: {token_savings:.1f}%")
print(f"  Efficiency improvement: {efficiency_improvement:+.1f}%")

# Route distribution for adaptive
route_dist = adaptive['route_distribution']
total_routes = sum(route_dist.values())
print(f"\n📍 ADAPTIVE ROUTING DISTRIBUTION:")
for route, count in route_dist.items():
    pct = 100 * count / total_routes
    print(f"  {route}: {count} ({pct:.1f}%)")

## 8. Visualizations

### 8.1 Accuracy vs Token Efficiency

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Accuracy comparison
strategies = [r['strategy'] for r in all_results]
accuracies = [r['accuracy'] * 100 for r in all_results]
colors = ['#3498db', '#e74c3c', '#95a5a6', '#2ecc71']

bars1 = ax1.bar(strategies, accuracies, color=colors, alpha=0.8)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
ax1.set_ylim(0, 100)
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1f}%',
             ha='center', va='bottom', fontsize=10)

# Plot 2: Token efficiency
avg_tokens = [r['avg_tokens'] for r in all_results]

bars2 = ax2.bar(strategies, avg_tokens, color=colors, alpha=0.8)
ax2.set_ylabel('Avg Tokens per Query', fontsize=12)
ax2.set_title('Token Usage Comparison', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.0f}',
             ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['results_dir'], 'accuracy_tokens_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: accuracy_tokens_comparison.png")

### 8.2 Efficiency Scatter Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for i, result in enumerate(all_results):
    ax.scatter(
        result['avg_tokens'],
        result['accuracy'] * 100,
        s=300,
        c=colors[i],
        alpha=0.7,
        edgecolors='black',
        linewidth=2,
        label=result['strategy']
    )
    
    # Add strategy label
    ax.annotate(
        result['strategy'],
        (result['avg_tokens'], result['accuracy'] * 100),
        xytext=(10, 10),
        textcoords='offset points',
        fontsize=11,
        fontweight='bold'
    )

ax.set_xlabel('Average Tokens per Query', fontsize=13)
ax.set_ylabel('Accuracy (%)', fontsize=13)
ax.set_title('Accuracy vs Token Efficiency\n(Upper-left is better: high accuracy, low tokens)',
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['results_dir'], 'efficiency_scatter.png'), dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: efficiency_scatter.png")

### 8.3 Entropy Distribution for Adaptive Router

In [ ]:
# Extract entropy values and routes
adaptive_results = results_adaptive['results']
entropies_direct = [r['entropy'] for r in adaptive_results if r['route_taken'] == 'direct' and not np.isnan(r['entropy'])]
entropies_cot = [r['entropy'] for r in adaptive_results if r['route_taken'] == 'cot' and not np.isnan(r['entropy'])]

fig, ax = plt.subplots(figsize=(12, 6))

# Plot histograms
ax.hist(entropies_direct, bins=30, alpha=0.6, color='#3498db', label=f'Direct (n={len(entropies_direct)})', edgecolor='black')
ax.hist(entropies_cot, bins=30, alpha=0.6, color='#e74c3c', label=f'CoT (n={len(entropies_cot)})', edgecolor='black')

# Add threshold lines
ax.axvline(CONFIG['entropy_threshold_low'], color='green', linestyle='--', linewidth=2, label=f'Low threshold ({CONFIG["entropy_threshold_low"]})')
ax.axvline(CONFIG['entropy_threshold_high'], color='orange', linestyle='--', linewidth=2, label=f'High threshold ({CONFIG["entropy_threshold_high"]})')

ax.set_xlabel('Intention Entropy (bits)', fontsize=13)
ax.set_ylabel('Frequency', fontsize=13)
ax.set_title('Entropy Distribution by Route Decision', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['results_dir'], 'entropy_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: entropy_distribution.png")
print(f"\nEntropy statistics:")
print(f"  Direct: mean={np.mean(entropies_direct):.3f}, std={np.std(entropies_direct):.3f}")
print(f"  CoT: mean={np.mean(entropies_cot):.3f}, std={np.std(entropies_cot):.3f}")

### 8.4 Correctness by Entropy

In [ ]:
# Bin problems by entropy
adaptive_results_df = pd.DataFrame(adaptive_results)
adaptive_results_df = adaptive_results_df[~adaptive_results_df['entropy'].isna()]

# Create entropy bins
adaptive_results_df['entropy_bin'] = pd.cut(
    adaptive_results_df['entropy'],
    bins=[0, 0.3, 0.5, 0.8, 1.2, 2.0],
    labels=['0-0.3', '0.3-0.5', '0.5-0.8', '0.8-1.2', '1.2+']
)

# Calculate accuracy per bin
accuracy_by_bin = adaptive_results_df.groupby('entropy_bin')['is_correct'].agg(['mean', 'count'])
accuracy_by_bin['mean'] *= 100  # Convert to percentage

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(range(len(accuracy_by_bin)), accuracy_by_bin['mean'], color='#2ecc71', alpha=0.7, edgecolor='black')
ax.set_xticks(range(len(accuracy_by_bin)))
ax.set_xticklabels(accuracy_by_bin.index, fontsize=11)
ax.set_xlabel('Entropy Bin (bits)', fontsize=13)
ax.set_ylabel('Accuracy (%)', fontsize=13)
ax.set_title('Accuracy vs Intention Entropy', fontsize=14, fontweight='bold')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for i, (bar, count) in enumerate(zip(bars, accuracy_by_bin['count'])):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%\n(n={int(count)})',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['results_dir'], 'accuracy_by_entropy.png'), dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: accuracy_by_entropy.png")

## 9. Detailed Analysis

### 9.1 Router Decision Analysis

In [ ]:
# Analyze router decisions
adaptive_df = pd.DataFrame(adaptive_results)

print("\n" + "="*80)
print("ROUTER DECISION ANALYSIS")
print("="*80)

# Accuracy by route
print("\n1. Accuracy by Route:")
accuracy_by_route = adaptive_df.groupby('route_taken')['is_correct'].agg(['mean', 'count'])
accuracy_by_route['mean'] *= 100
print(accuracy_by_route.to_string())

# Token usage by route
print("\n2. Token Usage by Route:")
tokens_by_route = adaptive_df.groupby('route_taken')['total_tokens'].agg(['mean', 'median', 'std'])
print(tokens_by_route.to_string())

# Entropy statistics by correctness
print("\n3. Entropy Statistics by Correctness:")
entropy_by_correctness = adaptive_df.groupby('is_correct')['entropy'].agg(['mean', 'median', 'std'])
entropy_by_correctness.index = ['Incorrect', 'Correct']
print(entropy_by_correctness.to_string())

# Correlation between entropy and correctness
from scipy import stats
correlation, p_value = stats.pointbiserialr(adaptive_df['is_correct'], adaptive_df['entropy'])
print(f"\n4. Correlation between Entropy and Correctness:")
print(f"   Pearson r: {correlation:.3f}")
print(f"   P-value: {p_value:.4f}")
if p_value < 0.05:
    print(f"   → Statistically significant correlation!")
else:
    print(f"   → Not statistically significant")

### 9.2 Error Analysis

In [ ]:
print("\n" + "="*80)
print("ERROR ANALYSIS")
print("="*80)

# Get incorrect predictions
errors = adaptive_df[~adaptive_df['is_correct']].copy()
errors = errors.sort_values('entropy')

print(f"\nTotal errors: {len(errors)} / {len(adaptive_df)} ({len(errors)/len(adaptive_df)*100:.1f}%)")

# Errors by route
print("\nErrors by Route:")
errors_by_route = errors['route_taken'].value_counts()
for route, count in errors_by_route.items():
    total_route = len(adaptive_df[adaptive_df['route_taken'] == route])
    error_rate = count / total_route * 100
    print(f"  {route}: {count} errors / {total_route} total ({error_rate:.1f}% error rate)")

# Show sample errors
print("\nSample Errors (5 lowest entropy, 5 highest entropy):")
print("\nLow Entropy Errors (model was confident but wrong):")
for idx in errors.head(5).index:
    row = errors.loc[idx]
    print(f"\n  Problem {row['problem_idx']}:")
    print(f"    Question: {row['question'][:80]}...")
    print(f"    Ground truth: {row['ground_truth']}")
    print(f"    Model answer: {row['extracted_answer']}")
    print(f"    Entropy: {row['entropy']:.3f}")
    print(f"    Route: {row['route_taken']}")

print("\nHigh Entropy Errors (model was uncertain and wrong):")
for idx in errors.tail(5).index:
    row = errors.loc[idx]
    print(f"\n  Problem {row['problem_idx']}:")
    print(f"    Question: {row['question'][:80]}...")
    print(f"    Ground truth: {row['ground_truth']}")
    print(f"    Model answer: {row['extracted_answer']}")
    print(f"    Entropy: {row['entropy']:.3f}")
    print(f"    Route: {row['route_taken']}")

## 10. Save Results

In [ ]:
print("\nSaving results...")

# Save summary
summary_file = os.path.join(CONFIG['results_dir'], 'experiment_summary.json')
with open(summary_file, 'w') as f:
    json.dump({
        'config': CONFIG,
        'timestamp': datetime.now().isoformat(),
        'model_info': {
            'name': CONFIG['model_name'],
            'parameters': f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B"
        },
        'results': {
            'always_direct': results_always_direct['summary'],
            'always_cot': results_always_cot['summary'],
            'random': results_random_dict['summary'],
            'adaptive': results_adaptive['summary']
        }
    }, f, indent=2)
print(f"  ✓ Saved summary: {summary_file}")

# Save detailed results
for name, results in [
    ('always_direct', results_always_direct),
    ('always_cot', results_always_cot),
    ('random', results_random_dict),
    ('adaptive', results_adaptive)
]:
    detail_file = os.path.join(CONFIG['results_dir'], f'{name}_detailed.json')
    with open(detail_file, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"  ✓ Saved detailed results: {detail_file}")

# Save comparison DataFrame
csv_file = os.path.join(CONFIG['results_dir'], 'comparison.csv')
df_comparison.to_csv(csv_file, index=False)
print(f"  ✓ Saved comparison CSV: {csv_file}")

print("\n✅ All results saved successfully!")

## 11. Conclusions

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT CONCLUSIONS")
print("="*80)

adaptive_summary = results_adaptive['summary']
cot_summary = results_always_cot['summary']
direct_summary = results_always_direct['summary']

print("\n🎯 KEY FINDINGS:")
print("\n1. Accuracy:")
print(f"   - Adaptive Router: {adaptive_summary['accuracy']:.1%}")
print(f"   - Always-CoT: {cot_summary['accuracy']:.1%}")
print(f"   - Always-Direct: {direct_summary['accuracy']:.1%}")
accuracy_vs_cot = (adaptive_summary['accuracy'] - cot_summary['accuracy']) * 100
print(f"   → Adaptive is {accuracy_vs_cot:+.1f} pp vs Always-CoT")

print("\n2. Token Efficiency:")
print(f"   - Adaptive Router: {adaptive_summary['avg_tokens']:.1f} tokens/query")
print(f"   - Always-CoT: {cot_summary['avg_tokens']:.1f} tokens/query")
print(f"   - Always-Direct: {direct_summary['avg_tokens']:.1f} tokens/query")
token_savings = (1 - adaptive_summary['total_tokens'] / cot_summary['total_tokens']) * 100
print(f"   → Adaptive saves {token_savings:.1f}% tokens vs Always-CoT")

print("\n3. Overall Efficiency (Accuracy per 1k tokens):")
print(f"   - Adaptive Router: {adaptive_summary['efficiency']:.4f}")
print(f"   - Always-CoT: {cot_summary['efficiency']:.4f}")
print(f"   - Always-Direct: {direct_summary['efficiency']:.4f}")
efficiency_improvement = (adaptive_summary['efficiency'] / cot_summary['efficiency'] - 1) * 100
print(f"   → Adaptive is {efficiency_improvement:+.1f}% more efficient than Always-CoT")

print("\n4. Routing Behavior:")
route_dist = adaptive_summary['route_distribution']
total_routes = sum(route_dist.values())
for route, count in route_dist.items():
    pct = 100 * count / total_routes
    print(f"   - {route.capitalize()}: {pct:.1f}% ({count} problems)")

print("\n💡 INTERPRETATION:")
if accuracy_vs_cot >= -2 and token_savings > 20:
    print("   ✅ Adaptive router achieves comparable accuracy with significant token savings!")
    print("   ✅ This demonstrates effective cost-performance trade-off.")
elif accuracy_vs_cot > 2:
    print("   ✅ Adaptive router improves accuracy AND saves tokens!")
    print("   ✅ This is the best case scenario - pareto improvement.")
else:
    print("   ⚠️  Adaptive router shows trade-offs between accuracy and efficiency.")
    print("   → Consider adjusting thresholds or exploring other routing strategies.")

print("\n📊 STATISTICAL SIGNIFICANCE:")
# Simple z-test for proportion difference
n = adaptive_summary['n_problems']
p1 = adaptive_summary['accuracy']
p2 = cot_summary['accuracy']
pooled_p = (p1 + p2) / 2
se = np.sqrt(pooled_p * (1 - pooled_p) * (2 / n))
z_score = (p1 - p2) / se
from scipy.stats import norm
p_value = 2 * (1 - norm.cdf(abs(z_score)))

print(f"   Z-score: {z_score:.3f}")
print(f"   P-value: {p_value:.4f}")
if p_value < 0.05:
    print(f"   → Statistically significant difference (p < 0.05)")
else:
    print(f"   → Not statistically significant (p >= 0.05)")

print("\n" + "="*80)

## 12. Next Steps

Based on these results, consider:

1. **Threshold Tuning**: Experiment with different entropy thresholds to optimize the accuracy/efficiency trade-off
2. **Error Analysis**: Investigate low-entropy errors (confident but wrong) to improve routing decisions
3. **Extended Evaluation**: Test on ARC-Challenge and AQUA with option-normalized entropy and constrained decoding
4. **Self-Consistency**: For high-entropy problems, use self-consistency (Task 6) instead of single CoT
5. **Hybrid Strategies**: Combine entropy-based routing with probe-based early exit